# 🗄️ SQL & SQLite with Python
### Complete Beginner-Friendly Notes
---
> **Topics Covered:** SQL vs SQLite • Connect • Create Table • INSERT • SELECT • UPDATE • DELETE • Bulk Insert (executemany) • Close Connection

---

## 📌 What is SQL vs SQLite?

| Feature | SQL (MySQL/PostgreSQL) | SQLite |
|---------|----------------------|--------|
| Type | Full database server | Lightweight file-based DB |
| Server needed? | Yes | No (serverless) |
| Configuration | Complex setup | Zero configuration |
| Storage | Separate server | Single `.db` file on disk |
| Best for | Large enterprise apps | Embedded systems, small apps, learning |
| Python built-in? | No (need connector) | **Yes** (`import sqlite3`) |

> **Simple way to think:** SQL is a big restaurant with staff, kitchen, servers. SQLite is a tiffin box you carry yourself — everything in one file!

---

## ⚙️ SECTION 0: Import SQLite3


In [ ]:
# SQLite3 comes built-in with Python 3 — no pip install needed!
import sqlite3

print('✅ sqlite3 imported!')
print('SQLite version:', sqlite3.sqlite_version)

---
## 🔌 SECTION 1: Connect to a Database

SQLite database = simply a **`.db` file** on your computer.
When you run `connect()`, Python either **opens** the file if it exists, or **creates** it if it doesn't.


In [ ]:
# Connect to (or create) a database file
connection = sqlite3.connect('example.db')
# 'example.db' will be created in the current directory

print('✅ Connected to database!')
print(type(connection))   # <class 'sqlite3.Connection'>

### 💡 In-Memory Database (for testing):
```python
connection = sqlite3.connect(':memory:')  # Stored in RAM, deleted when program ends
```

---

## 🖱️ SECTION 2: Create a Cursor Object

**Cursor** = the tool that actually runs SQL commands and iterates through results.

Think of it like a **pen** — connection is the notebook, cursor is the pen you use to write/read.


In [ ]:
# Create a cursor object
cursor = connection.cursor()

print('✅ Cursor created!')
print(type(cursor))   # <class 'sqlite3.Cursor'>

### 📋 Connection vs Cursor:
| Object | Role |
|--------|------|
| `connection` | Links Python to the database file |
| `cursor` | Executes SQL commands & fetches results |

---

## 🏗️ SECTION 3: CREATE TABLE

Now let's create a table inside our database. Use `IF NOT EXISTS` to avoid errors on re-runs.


In [ ]:
# Create 'employees' table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS employees (
        id         INTEGER PRIMARY KEY,
        name       TEXT NOT NULL,
        age        INTEGER,
        department TEXT
    )
''')

# Commit the changes to save them permanently
connection.commit()

print('✅ Table created successfully!')

### 💡 Key Points:
- `IF NOT EXISTS` → Won't throw error if table already exists
- `PRIMARY KEY` → Unique identifier for each row (auto-increment)
- `NOT NULL` → This field CANNOT be left empty
- `connection.commit()` → **Always commit** after any write operation!

### SQLite Data Types:
| SQLite Type | Stores |
|-------------|--------|
| `INTEGER` | Whole numbers |
| `TEXT` | Strings/text |
| `REAL` | Decimal numbers |
| `BLOB` | Binary data (images, files) |
| `NULL` | Empty/missing value |

---

## ➕ SECTION 4: INSERT Data (C in CRUD)

Insert records one by one using `cursor.execute()`.


In [ ]:
# Insert individual records
cursor.execute("INSERT INTO employees (name, age, department) VALUES ('Krish', 32, 'Data Science')")
cursor.execute("INSERT INTO employees (name, age, department) VALUES ('Bob', 25, 'Engineering')")
cursor.execute("INSERT INTO employees (name, age, department) VALUES ('Charlie', 35, 'Finance')")

# Commit after all inserts
connection.commit()

print('✅ Records inserted!')

> ⚠️ **ALWAYS `connection.commit()` after INSERT/UPDATE/DELETE** — without it, changes won't be saved!

---

## 🔍 SECTION 5: SELECT / READ Data (R in CRUD)

Query data using `cursor.execute()` + fetch methods.


In [ ]:
# Execute SELECT query
cursor.execute('SELECT * FROM employees')

# Fetch ALL rows at once
rows = cursor.fetchall()

# Print each row
print('All employees:')
for row in rows:
    print(row)  # Each row = a tuple: (id, name, age, department)

### 📋 Fetch Methods:
| Method | What it returns |
|--------|----------------|
| `cursor.fetchall()` | All rows as a list of tuples |
| `cursor.fetchone()` | Only the first/next row |
| `cursor.fetchmany(n)` | Next n rows |

```python
# Example: SELECT with WHERE condition
cursor.execute("SELECT * FROM employees WHERE department = 'Engineering'")
print(cursor.fetchall())
```

---

## ✏️ SECTION 6: UPDATE Data (U in CRUD)

Modify existing records using the `UPDATE` SQL command.


In [ ]:
# Update Krish's age from 32 to 34
cursor.execute("""
    UPDATE employees
    SET age = 34
    WHERE name = 'Krish'
""")

connection.commit()   # Don't forget to commit!

# Verify the update
cursor.execute('SELECT * FROM employees')
for row in cursor.fetchall():
    print(row)

> ⚠️ **Always use WHERE in UPDATE** — without it, ALL rows will be updated!
> ```python
> # DANGEROUS — updates every row's age!
> cursor.execute('UPDATE employees SET age = 34')
> ```

---

## 🗑️ SECTION 7: DELETE Data (D in CRUD)

Remove specific records using the `DELETE FROM` command.


In [ ]:
# Delete Bob's record
cursor.execute("""
    DELETE FROM employees
    WHERE name = 'Bob'
""")

connection.commit()

# Verify deletion — should now show only 2 records
cursor.execute('SELECT * FROM employees')
print('After deleting Bob:')
for row in cursor.fetchall():
    print(row)

> ⚠️ **Always use WHERE in DELETE** — without it, ALL rows will be deleted!
> ```python
> cursor.execute('DELETE FROM employees')  # WIPES entire table!
> ```

---

## 🚀 SECTION 8: Bulk Insert — executemany()

Instead of calling `execute()` 100 times in a loop, use `executemany()` — insert ALL records in one shot!


In [ ]:
# Create a sales table
connection2 = sqlite3.connect('sales_data.db')
cursor2 = connection2.cursor()

cursor2.execute('''
    CREATE TABLE IF NOT EXISTS sales (
        id      INTEGER PRIMARY KEY,
        date    TEXT NOT NULL,
        product TEXT NOT NULL,
        sales   INTEGER,
        region  TEXT
    )
''')

# Multiple records as list of tuples
sales_data = [
    ('2024-01-01', 'Laptop',  1200, 'North America'),
    ('2024-01-02', 'Phone',    800, 'Asia'),
    ('2024-01-03', 'Tablet',   600, 'Europe'),
    ('2024-01-04', 'Laptop',  1500, 'North America'),
    ('2024-01-05', 'Phone',    950, 'Asia'),
]

# executemany — inserts ALL records in one call
# '?' are placeholders — replaced by each tuple's values
cursor2.executemany("""
    INSERT INTO sales (date, product, sales, region)
    VALUES (?, ?, ?, ?)
""", sales_data)

connection2.commit()
print(f'✅ {len(sales_data)} records inserted using executemany!')

In [ ]:
# Verify all records
cursor2.execute('SELECT * FROM sales')
rows = cursor2.fetchall()
for row in rows:
    print(row)

### 💡 execute() vs executemany():
| Method | Use Case |
|--------|----------|
| `cursor.execute()` | Single SQL statement |
| `cursor.executemany()` | Same statement with multiple data rows |

### How `?` Placeholders Work:
```python
# WRONG (SQL Injection risk!) — never use f-strings for user data:
cursor.execute(f"INSERT INTO sales VALUES ('{name}', {age})")

# CORRECT — use ? placeholders:
cursor.execute('INSERT INTO employees (name, age) VALUES (?, ?)', ('Alice', 28))
```

---

## 🔒 SECTION 9: Close the Connection

When you're done, always close the connection to **free resources** and **prevent data corruption**.


In [ ]:
# Close connections
connection.close()
connection2.close()

print('✅ Connections closed!')

# Trying to use cursor after closing will raise an error:
# cursor.execute('SELECT * FROM employees')
# → ProgrammingError: Cannot operate on a closed database.

### 💡 Best Practice — Use `with` statement (auto-closes):
```python
with sqlite3.connect('example.db') as conn:
    cur = conn.cursor()
    cur.execute('SELECT * FROM employees')
    print(cur.fetchall())
# Connection automatically closed when 'with' block ends
```

---

## 🌍 SECTION 10: Full CRUD Project — Employee Manager

Complete example putting everything together:


In [ ]:
import sqlite3

# ── SETUP ──────────────────────────────────────────────
conn = sqlite3.connect(':memory:')   # In-memory DB for demo
cur  = conn.cursor()

# ── CREATE ─────────────────────────────────────────────
cur.execute('''
    CREATE TABLE IF NOT EXISTS employees (
        id         INTEGER PRIMARY KEY,
        name       TEXT NOT NULL,
        age        INTEGER,
        department TEXT
    )
''')

# ── INSERT (bulk) ──────────────────────────────────────
employees = [
    ('Krish',   32, 'Data Science'),
    ('Bob',     25, 'Engineering'),
    ('Charlie', 35, 'Finance'),
    ('Alice',   28, 'Marketing'),
]
cur.executemany('INSERT INTO employees (name, age, department) VALUES (?, ?, ?)', employees)
conn.commit()
print('--- After INSERT ---')
cur.execute('SELECT * FROM employees')
for r in cur.fetchall(): print(r)

# ── UPDATE ─────────────────────────────────────────────
cur.execute("UPDATE employees SET age = 34 WHERE name = 'Krish'")
conn.commit()
print('\n--- After UPDATE (Krish age 32→34) ---')
cur.execute('SELECT * FROM employees')
for r in cur.fetchall(): print(r)

# ── DELETE ─────────────────────────────────────────────
cur.execute("DELETE FROM employees WHERE name = 'Bob'")
conn.commit()
print('\n--- After DELETE (Bob removed) ---')
cur.execute('SELECT * FROM employees')
for r in cur.fetchall(): print(r)

# ── CLOSE ──────────────────────────────────────────────
conn.close()
print('\n✅ All CRUD operations done!')

---
# 🚀 SQLite3 Cheatsheet & Summary Sheet


```python
import sqlite3

# ── CONNECT & SETUP ──────────────────────────────────────
conn   = sqlite3.connect('mydb.db')   # Open/create .db file
cursor = conn.cursor()                 # Create cursor to run SQL

# ── CREATE TABLE ─────────────────────────────────────────
cursor.execute('''
    CREATE TABLE IF NOT EXISTS tablename (
        id   INTEGER PRIMARY KEY,
        col1 TEXT NOT NULL,
        col2 INTEGER
    )
''')
conn.commit()

# ── INSERT (single) ──────────────────────────────────────
cursor.execute("INSERT INTO tablename (col1, col2) VALUES (?, ?)", ('Alice', 25))
conn.commit()

# ── INSERT (bulk) ────────────────────────────────────────
data = [('Bob', 30), ('Charlie', 22)]
cursor.executemany('INSERT INTO tablename (col1, col2) VALUES (?, ?)', data)
conn.commit()

# ── SELECT (read) ────────────────────────────────────────
cursor.execute('SELECT * FROM tablename')
rows = cursor.fetchall()               # All rows
row  = cursor.fetchone()               # One row
for row in rows: print(row)

# ── UPDATE ───────────────────────────────────────────────
cursor.execute("UPDATE tablename SET col2 = 99 WHERE col1 = 'Alice'")
conn.commit()

# ── DELETE ───────────────────────────────────────────────
cursor.execute("DELETE FROM tablename WHERE col1 = 'Bob'")
conn.commit()

# ── CLOSE ────────────────────────────────────────────────
conn.close()
```


---
# ❓ Important Questions & Answers


**Q1. SQL aur SQLite mein kya fark hai?**
> SQL ek language hai (MySQL, PostgreSQL) jo server pe run karta hai aur heavy configuration chahiye. SQLite ek lightweight, serverless, file-based database engine hai — ek `.db` file mein sab kuch store hota hai.

**Q2. SQLite import karne ke liye kya karna padta hai?**
> Kuch nahi! `import sqlite3` — Python 3 ke saath by default aata hai, koi pip install nahi.

**Q3. Connection aur Cursor mein kya fark hai?**
> Connection = database file se link. Cursor = actual SQL commands run karne ka tool. Dono chahiye.

**Q4. `connection.commit()` kyun zaruri hai?**
> Bina commit ke changes RAM mein rehte hain, file mein save nahi hote. INSERT/UPDATE/DELETE ke baad ALWAYS commit karo.

**Q5. `CREATE TABLE IF NOT EXISTS` kyun use karte hain?**
> Code dobara run karne pe error nahi aayega. Agar table pehle se exist karta hai toh ignore kar deta hai.

**Q6. `cursor.fetchall()` vs `cursor.fetchone()` mein kya fark hai?**
> `fetchall()` = sab rows ek saath list mein. `fetchone()` = sirf pehli/next ek row.

**Q7. `execute()` vs `executemany()` mein kya fark hai?**
> `execute()` = ek statement ek baar. `executemany()` = same statement ko multiple data rows ke saath bulk mein run karo — much faster!

**Q8. `?` placeholder kya hota hai SQLite mein?**
> Safe way to pass values into SQL query. `cursor.execute('INSERT INTO t VALUES (?,?)', ('Alice', 25))`. Direct f-strings use karna SQL injection attack ka risk hai.

**Q9. PRIMARY KEY kya karta hai?**
> Har row ko ek unique identifier deta hai. SQLite mein INTEGER PRIMARY KEY = auto-increment (automatically 1, 2, 3... assign hota hai).

**Q10. `NOT NULL` constraint kya karta hai?**
> Us column mein NULL/empty value allowed nahi hogi — insert pe error aayega agar value missing ho.

**Q11. Connection close kyun karte hain?**
> Resources free karne ke liye. Close ke baad cursor use karne pe `ProgrammingError: Cannot operate on a closed database` error aata hai.

**Q12. In-memory database kab use karte hain?**
> Testing ya temporary operations ke liye: `sqlite3.connect(':memory:')`. Program khatam hote hi data delete ho jaata hai.

**Q13. UPDATE bina WHERE ke kya hoga?**
> TABLE KI SAARI ROWS UPDATE HO JAAYENGI! `UPDATE employees SET age = 0` → har employee ki age 0 ho jaayegi. ALWAYS use WHERE!

**Q14. DELETE bina WHERE ke kya hoga?**
> POORA TABLE EMPTY HO JAAYEGA! ALWAYS WHERE use karo. `DROP TABLE tablename` use karo sirf jab table hi delete karni ho.

**Q15. CRUD kya hota hai?**
> **C**reate (INSERT), **R**ead (SELECT), **U**pdate (UPDATE), **D**elete (DELETE) — 4 basic database operations.
